# Reranker Step 1 — Candidate Generation


## 1. Configuration

In [40]:
import os

# ── Paths ──────────────────────────────────────────────────────────────
PHASE1_DIR = "phase1_outputs"
PHASE2_DIR = "phase2_outputs"
PHASE3_DIR = "phase3_outputs"
os.makedirs(PHASE3_DIR, exist_ok=True)

TRAIN_FILE        = os.path.join(PHASE1_DIR, "interactions_model_ready_train.parquet")
VALID_FILE        = os.path.join(PHASE1_DIR, "interactions_model_ready_valid.parquet")
TEST_FILE         = os.path.join(PHASE1_DIR, "interactions_model_ready_test.parquet")
CHECKPOINT_FILE   = os.path.join(PHASE2_DIR, "lightgcn_best.pt")
VALID_SAMPLE_FILE = os.path.join(PHASE2_DIR, "lightgcn_valid_eval_users.npy")
TEST_SAMPLE_FILE  = os.path.join(PHASE2_DIR, "lightgcn_test_eval_users.npy")

VALID_CANDIDATES_FILE = os.path.join(PHASE3_DIR, "valid_candidates.parquet")
TEST_CANDIDATES_FILE  = os.path.join(PHASE3_DIR, "test_candidates.parquet")

# ── Model hyperparameters (MUST match the checkpoint) ──────────────────
EMBED_DIM   = 64
NUM_LAYERS  = 2

# ── Candidate generation ───────────────────────────────────────────────
K           = 1000       # candidates per user
EVAL_CHUNK  = 4096      # users per GPU scoring batch
SEED        = 42


## 2. Imports and device setup

In [41]:
import time
import numpy as np
import pandas as pd
import torch
from torch_geometric.nn.models import LightGCN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

def report_mem(label=""):
    """Quick CPU+GPU memory footprint helper."""
    try:
        import psutil
        rss = psutil.Process().memory_info().rss / 1024**3
        msg = f"[mem{(' ' + label) if label else ''}]  RAM: {rss:.2f} GB"
    except Exception:
        msg = f"[mem{(' ' + label) if label else ''}]"
    if torch.cuda.is_available():
        gpu = torch.cuda.memory_allocated() / 1024**3
        msg += f"  GPU: {gpu:.2f} GB"
    print(msg)

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
report_mem("start")


Device: cuda
GPU: NVIDIA GeForce RTX 5090
GPU memory: 31.8 GB
[mem start]  RAM: 8.80 GB  GPU: 1.63 GB


## 3. Verify required files exist

In [42]:
required = {
    "train parquet":   TRAIN_FILE,
    "valid parquet":   VALID_FILE,
    "test parquet":    TEST_FILE,
    "checkpoint":      CHECKPOINT_FILE,
    "valid users":     VALID_SAMPLE_FILE,
    "test users":      TEST_SAMPLE_FILE,
}
missing = [(name, path) for name, path in required.items() if not os.path.exists(path)]
if missing:
    msg = "Missing required files:\n" + "\n".join(f"  {n}: {p}" for n, p in missing)
    raise FileNotFoundError(msg)

print("All required files present:")
for name, path in required.items():
    size_mb = os.path.getsize(path) / 1024**2
    print(f"  {name:18s}: {path} ({size_mb:.1f} MB)")


All required files present:
  train parquet     : phase1_outputs\interactions_model_ready_train.parquet (818.6 MB)
  valid parquet     : phase1_outputs\interactions_model_ready_valid.parquet (124.9 MB)
  test parquet      : phase1_outputs\interactions_model_ready_test.parquet (119.9 MB)
  checkpoint        : phase2_outputs\lightgcn_best.pt (546.3 MB)
  valid users       : phase2_outputs\lightgcn_valid_eval_users.npy (0.4 MB)
  test users        : phase2_outputs\lightgcn_test_eval_users.npy (0.2 MB)


## 4. Load splits and rebuild graph structure

In [43]:
train_df = pd.read_parquet(TRAIN_FILE, columns=["user_idx", "item_idx", "rating"])
valid_df = pd.read_parquet(VALID_FILE, columns=["user_idx", "item_idx", "rating"])
test_df  = pd.read_parquet(TEST_FILE,  columns=["user_idx", "item_idx", "rating"])

# Node counts — global max across splits (same as LightGCN notebook)
n_users = int(max(
    train_df["user_idx"].max(),
    valid_df["user_idx"].max(),
    test_df["user_idx"].max(),
)) + 1
n_items = int(max(
    train_df["item_idx"].max(),
    valid_df["item_idx"].max(),
    test_df["item_idx"].max(),
)) + 1
n_nodes = n_users + n_items

print(f"Users:       {n_users:,}")
print(f"Items:       {n_items:,}")
print(f"Total nodes: {n_nodes:,}")
print(f"Train rows:  {len(train_df):,}")

# Rankable universe = items that appear in TRAIN
train_item_set = set(train_df["item_idx"].unique().tolist())
rankable_items_np = np.array(sorted(train_item_set), dtype=np.int64)
rankable_items_t  = torch.tensor(rankable_items_np, dtype=torch.long, device=device)
rankable_item_pos = {int(item): pos for pos, item in enumerate(rankable_items_np)}
print(f"Rankable items: {len(rankable_items_np):,}")

# Bidirectional bipartite edge index (same as LightGCN notebook)
train_users_np = train_df["user_idx"].to_numpy(dtype=np.int64)
train_items_np = train_df["item_idx"].to_numpy(dtype=np.int64)
src = np.concatenate([train_users_np, train_items_np + n_users])
dst = np.concatenate([train_items_np + n_users, train_users_np])
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long, device=device)
print(f"Edge index shape: {tuple(edge_index.shape)}")
report_mem("after load")


Users:       1,927,834
Items:       309,904
Total nodes: 2,237,738
Train rows:  16,064,988
Rankable items: 309,395
Edge index shape: (2, 32129976)
[mem after load]  RAM: 9.24 GB  GPU: 1.63 GB


## 5. Build seen-item masks

In [44]:
train_seen_by_user = train_df.groupby("user_idx")["item_idx"].agg(set).to_dict()

train_valid_df = pd.concat(
    [train_df[["user_idx", "item_idx"]], valid_df[["user_idx", "item_idx"]]],
    ignore_index=True,
)
train_valid_seen_by_user = train_valid_df.groupby("user_idx")["item_idx"].agg(set).to_dict()

def build_seen_positions(seen_by_user):
    out = {}
    for u, items in seen_by_user.items():
        pos_list = [rankable_item_pos[i] for i in items if i in rankable_item_pos]
        if pos_list:
            out[int(u)] = np.array(pos_list, dtype=np.int64)
    return out

train_seen_pos_by_user       = build_seen_positions(train_seen_by_user)
train_valid_seen_pos_by_user = build_seen_positions(train_valid_seen_by_user)

print(f"Users with train-seen mask:       {len(train_seen_pos_by_user):,}")
print(f"Users with train+valid seen mask: {len(train_valid_seen_pos_by_user):,}")
report_mem("after masks")


Users with train-seen mask:       1,927,834
Users with train+valid seen mask: 1,927,834
[mem after masks]  RAM: 9.27 GB  GPU: 1.63 GB


## 6. Load sampled user IDs and build target lookups

In [45]:
valid_users_raw = np.load(VALID_SAMPLE_FILE).tolist()
test_users_raw  = np.load(TEST_SAMPLE_FILE).tolist()
print(f"Sampled valid users: {len(valid_users_raw):,}")
print(f"Sampled test users:  {len(test_users_raw):,}")

def build_target_dict(eval_df, users_subset):
    users_set = set(users_subset)
    df = eval_df[eval_df["user_idx"].isin(users_set) & eval_df["item_idx"].isin(train_item_set)]
    return dict(zip(df["user_idx"].astype(int), df["item_idx"].astype(int)))

valid_target_by_user = build_target_dict(valid_df, valid_users_raw)
test_target_by_user  = build_target_dict(test_df,  test_users_raw)

valid_users = sorted(valid_target_by_user.keys())
test_users  = sorted(test_target_by_user.keys())

print(f"\nAfter warm-start filter:")
print(f"  valid users with target in train-item set: {len(valid_users):,} / {len(valid_users_raw):,}")
print(f"  test users with target in train-item set:  {len(test_users):,} / {len(test_users_raw):,}")


Sampled valid users: 50,000
Sampled test users:  25,000

After warm-start filter:
  valid users with target in train-item set: 50,000 / 50,000
  test users with target in train-item set:  25,000 / 25,000


## 7. Load the LightGCN checkpoint

In [46]:
model = LightGCN(
    num_nodes=n_nodes,
    embedding_dim=EMBED_DIM,
    num_layers=NUM_LAYERS,
).to(device)

model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=device))
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded checkpoint: {CHECKPOINT_FILE}")
print(f"Model parameters: {total_params:,}")
report_mem("after model load")


Loaded checkpoint: phase2_outputs\lightgcn_best.pt
Model parameters: 143,215,232
[mem after model load]  RAM: 9.77 GB  GPU: 1.63 GB


## 8. Precompute embeddings (one forward pass)

In [47]:
t0 = time.time()
with torch.no_grad():
    all_emb = model.get_embedding(edge_index)
    user_emb          = all_emb[:n_users]
    item_emb          = all_emb[n_users:n_users + n_items]
    rankable_item_emb = item_emb[rankable_items_t]

print(f"Forward pass over graph: {time.time() - t0:.1f}s")
print(f"user_emb shape:          {tuple(user_emb.shape)}")
print(f"rankable_item_emb shape: {tuple(rankable_item_emb.shape)}")
report_mem("after forward pass")


Forward pass over graph: 0.0s
user_emb shape:          (1927834, 64)
rankable_item_emb shape: (309395, 64)
[mem after forward pass]  RAM: 9.77 GB  GPU: 1.63 GB


## 9. Candidate generation function

For each chunk of users:
1. Score users against all rankable items via a single dot product.
2. Mask seen items (scores → −inf).
3. Take top-K scores and their indices.
4. Map local indices back to global `item_idx`.
5. Append to output arrays.


In [48]:
@torch.no_grad()
def generate_candidates(eval_users, target_by_user, seen_pos_by_user, label):
    """Generate top-K candidates for each user.

    Returns pd.DataFrame with columns:
        [user_idx, candidate_item_idx, lightgcn_score, rank, true_item_idx, is_positive]
    """
    chunks_users, chunks_items, chunks_scores, chunks_ranks = [], [], [], []
    rank_arr = np.arange(K, dtype=np.int32)

    t0 = time.time()
    n_chunks = (len(eval_users) + EVAL_CHUNK - 1) // EVAL_CHUNK

    for chunk_i, start in enumerate(range(0, len(eval_users), EVAL_CHUNK)):
        chunk = eval_users[start:start + EVAL_CHUNK]
        chunk_size = len(chunk)
        u_tensor = torch.tensor(chunk, dtype=torch.long, device=device)

        # Score: [chunk_size, n_rankable]
        scores = user_emb[u_tensor] @ rankable_item_emb.T

        # Mask seen items per user
        for local_i, u in enumerate(chunk):
            seen_pos = seen_pos_by_user.get(int(u))
            if seen_pos is not None and len(seen_pos) > 0:
                mask = torch.from_numpy(seen_pos).long().to(device)
                scores[local_i, mask] = -float("inf")

        # Top-K per user
        top_scores, top_local = scores.topk(K, dim=1)
        top_scores_np = top_scores.cpu().numpy()
        top_local_np  = top_local.cpu().numpy()
        top_global_np = rankable_items_np[top_local_np]

        chunks_users.append(np.repeat(np.array(chunk, dtype=np.int64), K))
        chunks_items.append(top_global_np.flatten().astype(np.int64))
        chunks_scores.append(top_scores_np.flatten().astype(np.float32))
        chunks_ranks.append(np.tile(rank_arr, chunk_size))

        if (chunk_i + 1) % 5 == 0 or chunk_i == n_chunks - 1:
            elapsed = time.time() - t0
            pct = 100 * (chunk_i + 1) / n_chunks
            print(f"  {label}: chunk {chunk_i+1}/{n_chunks} ({pct:.0f}%)  elapsed {elapsed:.0f}s")

    df = pd.DataFrame({
        "user_idx":            np.concatenate(chunks_users),
        "candidate_item_idx":  np.concatenate(chunks_items),
        "lightgcn_score":      np.concatenate(chunks_scores),
        "rank":                np.concatenate(chunks_ranks),
    })

    # Join target info
    target_df = pd.DataFrame({
        "user_idx":      list(target_by_user.keys()),
        "true_item_idx": list(target_by_user.values()),
    })
    df = df.merge(target_df, on="user_idx", how="left")
    df["is_positive"] = (df["candidate_item_idx"] == df["true_item_idx"]).astype(np.int8)

    print(f"  {label}: DONE in {time.time() - t0:.0f}s — {len(df):,} rows")
    return df


## 10. Generate candidates for both splits

Valid uses train-only seen masks; test uses train+valid seen masks.

In [49]:
print("Generating valid candidates...")
valid_candidates_df = generate_candidates(
    eval_users=valid_users,
    target_by_user=valid_target_by_user,
    seen_pos_by_user=train_seen_pos_by_user,
    label="valid",
)
print(f"Valid candidates shape: {valid_candidates_df.shape}\n")

print("Generating test candidates...")
test_candidates_df = generate_candidates(
    eval_users=test_users,
    target_by_user=test_target_by_user,
    seen_pos_by_user=train_valid_seen_pos_by_user,
    label="test",
)
print(f"Test candidates shape: {test_candidates_df.shape}")

valid_candidates_df.head()


Generating valid candidates...
  valid: chunk 5/13 (38%)  elapsed 1s
  valid: chunk 10/13 (77%)  elapsed 2s
  valid: chunk 13/13 (100%)  elapsed 3s
  valid: DONE in 4s — 50,000,000 rows
Valid candidates shape: (50000000, 6)

Generating test candidates...
  test: chunk 5/7 (71%)  elapsed 1s
  test: chunk 7/7 (100%)  elapsed 1s
  test: DONE in 2s — 25,000,000 rows
Test candidates shape: (25000000, 6)


,user_idx,candidate_item_idx,lightgcn_score,rank,true_item_idx,is_positive
0,1,27054,11.161092,0,211140,0
1,1,206993,10.363952,1,211140,0
2,1,26442,10.166111,2,211140,0
3,1,29405,9.834175,3,211140,0
4,1,269396,9.578410,4,211140,0


## 11. Save to parquet

In [50]:
valid_candidates_df.to_parquet(VALID_CANDIDATES_FILE, index=False)
test_candidates_df.to_parquet(TEST_CANDIDATES_FILE, index=False)

for path in [VALID_CANDIDATES_FILE, TEST_CANDIDATES_FILE]:
    size_mb = os.path.getsize(path) / 1024**2
    print(f"Saved {path}: {size_mb:.1f} MB")


Saved phase3_outputs\valid_candidates.parquet: 350.5 MB
Saved phase3_outputs\test_candidates.parquet: 175.4 MB


---

# Diagnostics

## Recall / HR / NDCG at various K

Recall@200 bounds everything the reranker can do. The gap between Recall@10 and Recall@200 is the reranking headroom.

In [51]:
RECALL_KS = [100, 200, 500, 1000]

def metric_table(candidates_df, split_name, ks):
    n_users = candidates_df["user_idx"].nunique()
    pos = candidates_df[candidates_df["is_positive"].astype(int) == 1].copy()
    if pos.empty:
        return pd.DataFrame({"split": [split_name]*len(ks), "K": ks,
                             "hits": [0]*len(ks), "users": [n_users]*len(ks),
                             "Recall/HR@K": [0.0]*len(ks), "NDCG@K": [0.0]*len(ks)})

    # Rank is 0-based, so NDCG discount = 1 / log2(rank + 2).
    pos["ndcg_discount"] = 1.0 / np.log2(pos["rank"].astype(float) + 2.0)
    rows = []
    for k in ks:
        pos_at_k = pos[pos["rank"] < k]
        hits = pos_at_k["user_idx"].nunique()
        rows.append({
            "split": split_name, "K": k, "hits": hits, "users": n_users,
            "Recall/HR@K": hits / n_users,
            "NDCG@K": float(pos_at_k["ndcg_discount"].sum() / n_users),
        })
    return pd.DataFrame(rows)

metrics = pd.concat([
    metric_table(valid_candidates_df, "valid", RECALL_KS),
    metric_table(test_candidates_df,  "test",  RECALL_KS),
], ignore_index=True)
print(metrics.round(4).to_string(index=False))


split    K  hits  users  Recall/HR@K  NDCG@K
valid  100  4441  50000       0.0888  0.0229
valid  200  6625  50000       0.1325  0.0290
valid  500 10638  50000       0.2128  0.0387
valid 1000 14747  50000       0.2949  0.0473
 test  100  2037  25000       0.0815  0.0209
 test  200  3054  25000       0.1222  0.0266
 test  500  4931  25000       0.1972  0.0356
 test 1000  6624  25000       0.2650  0.0428


## Positive-candidate summary

In [52]:
def positive_rank_summary(candidates_df, split_name):
    n_users = candidates_df["user_idx"].nunique()
    pos = candidates_df[candidates_df["is_positive"].astype(int) == 1].copy()
    retrieved = pos["user_idx"].nunique()
    missing = n_users - retrieved

    if pos.empty:
        return pd.DataFrame([{
            "split": split_name, "users": n_users,
            "users with positive candidate": 0,
            "users with no positive candidate": missing,
            "positive candidate rate": 0.0,
            "median positive rank": np.nan, "p90 positive rank": np.nan,
        }])

    rank_1_based = pos["rank"].astype(int) + 1
    return pd.DataFrame([{
        "split": split_name,
        "users": n_users,
        "users with positive candidate": retrieved,
        "users with no positive candidate": missing,
        "positive candidate rate": retrieved / n_users,
        "median positive rank": float(rank_1_based.median()),
        "p90 positive rank": float(rank_1_based.quantile(0.90)),
    }])

summary = pd.concat([
    positive_rank_summary(valid_candidates_df, "valid"),
    positive_rank_summary(test_candidates_df,  "test"),
], ignore_index=True)
print(summary.round(4).to_string(index=False))

print("\nInterpretation:")
print("- Recall@200 is the reranker's hard ceiling — missed targets cannot be recovered.")
print("- Gap between Recall@10 and Recall@200 is the reranking headroom.")
print("- Users with no positive candidate will be excluded from LambdaMART training in NB3.")


split  users  users with positive candidate  users with no positive candidate  positive candidate rate  median positive rank  p90 positive rank
valid  50000                          14747                             35253                   0.2949                 245.0              798.0
 test  25000                           6624                             18376                   0.2650                 232.0              782.4

Interpretation:
- Recall@200 is the reranker's hard ceiling — missed targets cannot be recovered.
- Gap between Recall@10 and Recall@200 is the reranking headroom.
- Users with no positive candidate will be excluded from LambdaMART training in NB3.
